# Code Attribution

Portions of this code are adapted from:

**FLASH: A Comprehensive Approach to Intrusion Detection via Provenance Graph Representation Learning**

Source: [https://github.com/DART-Laboratory/Flash-IDS](https://github.com/DART-Laboratory/Flash-IDS)


## Import required packages

In [ ]:
import pandas as pd
import numpy as np
import pickle
import torch
from torch_geometric.data import Data
import torch.nn.functional as F
import warnings
from tqdm import tqdm
import json
from torch_geometric.nn import SAGEConv
import math
from gensim.models import Word2Vec
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
import re
import gdown
import requests
import sys
import os
import glob
import gzip
import io
warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
%matplotlib inline

### Download the OpTC files from the FLASH Google drive

In [ ]:
def extract_logs(filepath, hostid):
    search_pattern = f'SysClient{hostid}'
    output_filename = f'SysClient{hostid}.systemia.com.txt'
    
    with gzip.open(filepath, 'rt', encoding='utf-8') as fin:
        with open(output_filename, 'ab') as f:
            out = io.BufferedWriter(f)
            for line in fin:
                if search_pattern in line:
                    out.write(line.encode('utf-8'))
            out.flush()

In [ ]:
def prepare_test_set():
    urls = [
        "https://drive.google.com/file/d/1HFSyvmgH0jvdnnnTdKfWRjZYOrLWoIkv/view?usp=drive_link",
        "https://drive.google.com/file/d/1pJLxJsDV8sngiedbfVajMetczIgM3PQd/view?usp=drive_link",
        "https://drive.google.com/file/d/1fRQqc68r8-z5BL7H_eAKIDOeHp7okDuM/view?usp=drive_link",
        "https://drive.google.com/file/d/1VfyGr8wfSe8LBIHBWuYBlU8c2CyEgO5C/view?usp=drive_link",
        "https://drive.google.com/file/d/10N9ZPolq_L8HivBqzf_jFKbwjSxddsZp/view?usp=drive_link",
        "https://drive.google.com/file/d/1xIr8gw-4zc8ESjUpYtrFsbOwhPGUSd15/view?usp=drive_link",
        "https://drive.google.com/file/d/1PvlCp2oQaxEBEFGSQWfcFVj19zLOe7yH/view?usp=drive_link"
    ]

    for url in urls:
        gdown.download(url, quiet=False, use_cookies=False, fuzzy=True)

    log_files = [
        ("AIA-201-225.ecar-2019-12-08T11-05-10.046.json.gz", "0201"),
        ("AIA-201-225.ecar-last.json.gz", "0201"),
        ("AIA-501-525.ecar-2019-11-17T04-01-58.625.json.gz", "0501"),
        ("AIA-501-525.ecar-last.json.gz", "0501"),
        ("AIA-51-75.ecar-last.json.gz", "0051")
    ]
    
    os.system("rm SysClient0201.com.txt")
    os.system("rm SysClient0501.com.txt")
    os.system("rm SysClient0051.com.txt")

    for file, code in tqdm(log_files, desc="Extracting logs", unit="file"):
        extract_logs(file, code)

prepare_test_set()

### Import required models

In [ ]:
# GitHub API URL for the folder
api_url = "https://api.github.com/repos/DART-Laboratory/Flash-IDS/contents/trained_weights/optc"

# Local folder to save the weights
os.makedirs("trained_weights/optc", exist_ok=True)

# Fetch the list of files from GitHub API
response = requests.get(api_url)
if response.status_code == 200:
    files = response.json()
    for file in files:
        if file["name"].endswith(".pth") or file["name"].endswith(".model") or file["name"].endswith(".pkl"):
            download_url = file["download_url"]
            save_path = os.path.join("trained_weights/optc", file["name"])

            print(f"Downloading {file['name']}...")
            r = requests.get(download_url)
            with open(save_path, "wb") as f:
                f.write(r.content)
            print(f"Saved: {save_path}")

    print("\nAll .pth and .model files downloaded successfully!")
else:
    print(f"Failed to access repo: {response.status_code}")

In [ ]:
url = "https://drive.usercontent.google.com/download?id=1x1IyE2L1X33DJpWwluMO8R4YR1W8mlb5&export=download&confirm=t&uuid=583dcb16-b3ce-4c67-a12c-fbf3e2c4b77e"
save_path = "trained_weights/optc/w2v_optc.model"

os.makedirs("trained_weights/optc", exist_ok=True)

print("Downloading w2v_optc.model...")
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    total = int(r.headers.get("Content-Length", 0))
    
    from tqdm import tqdm
    bar = tqdm(total=total, unit="B", unit_scale=True, desc="w2v_model.model")

    with open(save_path, "wb") as f:
        for chunk in r.iter_content(1024 * 1024):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

    bar.close()

print("Saved:", save_path)

In [ ]:
gnn_weights = "trained_weights/optc/gnn_temp.pth"
xgboost_weights = "trained_weights/optc/xgb.pkl"
word2vec_weights = "trained_weights/optc/w2v_optc.model"

### Download required variables from google drive

In [ ]:

file_id = "1G30P64AF1owrIBA-tEbysrjH4cFu04m1"
download_url = f"https://drive.google.com/uc?export=download&id={file_id}"


response = requests.get(download_url)
response.raise_for_status()

gnn_map = json.loads(response.text)


### Downloading the Ground truth from the FLASH github repository

In [ ]:
url = "https://raw.githubusercontent.com/DART-Laboratory/Flash-IDS/refs/heads/main/data_files/optc.txt"

# Step 1: Download the file
response = requests.get(url)

if response.status_code == 200:
    # Step 2: Save file in the current directory
    with open("optc.txt", "w") as f:
        f.write(response.text)

    print("optc.txt downloaded and saved to current directory!")
else:
    print(f"Failed to download: {response.status_code}")
    sys.exit(0)


### FLASH code for Graph creation

In [ ]:
# Functions to read the optc dataframe
def is_valid_entry(entry):
    valid_objects = {'PROCESS', 'FILE', 'FLOW', 'MODULE'}
    invalid_actions = {'START', 'TERMINATE'}

    object_valid = entry['object'] in valid_objects
    action_valid = entry['action'] not in invalid_actions
    actor_object_different = entry['actorID'] != entry['objectID']

    return object_valid and action_valid and actor_object_different

def Traversal_Rules(data):
    filtered_data = {}

    for entry in data:
        if is_valid_entry(entry):
            key = (
                entry['action'], 
                entry['actorID'], 
                entry['objectID'], 
                entry['object'], 
                entry['pid'], 
                entry['ppid']
            )
            filtered_data[key] = entry

    return list(filtered_data.values())

def Sentence_Construction(entry):
    action = entry["action"]
    properties = entry['properties']
    object_type = entry['object']

    format_strings = {
        'PROCESS': "{parent_image_path} {action} {image_path} {command_line}",
        'FILE': "{image_path} {action} {file_path}",
        'FLOW': "{image_path} {action} {src_ip} {src_port} {dest_ip} {dest_port} {direction}",
        'MODULE': "{image_path} {action} {module_path}"
    }

    default_format = "{image_path} {action} {module_path}"

    try:
        format_str = format_strings.get(object_type, default_format)
        phrase = format_str.format(action=action, **properties)
    except KeyError:
        phrase = ''

    return phrase.split(' ')

def Extract_Semantic_Info(event):
    object_type = event['object']
    properties = event['properties']

    label_mapping = {
        "PROCESS": ('parent_image_path', 'image_path'),
        "FILE": ('image_path', 'file_path'),
        "MODULE": ('image_path', 'module_path'),
        "FLOW": ('image_path', 'dest_ip', 'dest_port')
    }

    label_keys = label_mapping.get(object_type, None)
    if label_keys:
        labels = [properties.get(key) for key in label_keys]
        if all(labels):
            event["actorname"], event["objectname"] = labels[0], ' '.join(labels[1:])
            return event
    return None

def transform(text):
    labeled_data = [event for event in (Extract_Semantic_Info(x) for x in text) if event]
    data = Traversal_Rules(labeled_data)

    phrases = [Sentence_Construction(x) for x in data if Sentence_Construction(x)]
    for datum, phrase in zip(data, phrases):
        datum['phrase'] = phrase

    df = pd.DataFrame(data)
    s=0
    try:
        df['timestamp'] = pd.to_datetime(df['timestamp'].str[:-6], infer_datetime_format=True)
    except:
        s+=1
    df.sort_values(by='timestamp', inplace=True)

    return df

def Featurize(df):
    dummies = {'PROCESS': 0, 'FLOW': 1, 'FILE': 2, 'MODULE': 3}

    nodes = {}
    labels = {}
    lblmap = {}
    neimap = {}
    edges = []

    for index, row in df.iterrows():
        actor_id, object_id = row['actorID'], row["objectID"]
        object_type = row['object']

        nodes.setdefault(actor_id, []).extend(row['phrase'])
        nodes.setdefault(object_id, []).extend(row['phrase'])

        labels[actor_id] = dummies.get('PROCESS', -1)
        labels[object_id] = dummies.get(object_type, -1)

        lblmap[actor_id] = row['actorname']
        lblmap[object_id] = row['objectname']

        neimap.setdefault(actor_id, set()).add(row['objectname'])
        neimap.setdefault(object_id, set()).add(row['actorname'])

        edge_type = row['properties']['direction'] if object_type == 'FLOW' else row['action']
        edges.append((actor_id, object_id, edge_type))

    features, feat_labels, edge_index = [], [], [[], []]
    node_index = {}
    
    for node, phrases in nodes.items():
        if not (len(phrases) == 1 and phrases[0] == 'DELETE'):
            features.append(infer(phrases))
            feat_labels.append(labels[node])
            node_index[node] = len(features) - 1

    for src, dst, _ in edges:
        edge_index[0].append(node_index[src])
        edge_index[1].append(node_index[dst])

    mapp = list(node_index.keys())
    
    phrases_list = list(nodes.values())

    return features, np.array(feat_labels), edge_index, mapp, lblmap, neimap, phrases_list

def load_data(file_path):
    with open(file_path, 'r') as file:
        content = [json.loads(line) for line in file]
    
    return Featurize(transform(content))

In [ ]:
class PositionalEncoder:
    def __init__(self, d_model, max_len=100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)

    def embed(self, x):
        return x + self.pe[:x.size(0)]


def infer(document):
    word_embeddings = [w2vmodel.wv[word] for word in document if word in  w2vmodel.wv]
    
    if not word_embeddings:
        return np.zeros(20)

    output_embedding = torch.tensor(word_embeddings, dtype=torch.float)
    if len(document) < 100000:
        output_embedding = encoder.embed(output_embedding)

    output_embedding = output_embedding.detach().cpu().numpy()
    return np.mean(output_embedding, axis=0)

encoder = PositionalEncoder(20)
w2vmodel = Word2Vec.load(word2vec_weights)


In [ ]:
def load_features(filename=None, similarity=1):
    nodes, y_train, edges, mapp, lbl, nemap = load_data(filename)
    zero_vector = np.zeros(20)

    X_train = []
    for idx, map_item in enumerate(mapp):
        label = lbl[map_item]
        node_feature = nodes[idx]

        if label in gnn_map:
            emb, stored_set = gnn_map[label]
            current_set = nemap[map_item]
            jaccard_similarity = len(current_set.intersection(stored_set)) / len(current_set.union(stored_set))

            feature_vector = emb if jaccard_similarity >= similarity else zero_vector
        else:
            feature_vector = zero_vector

        X_train.append(np.hstack((node_feature, feature_vector)))

    return np.array(X_train), y_train, edges, mapp

### Evasion code applied in the modified load_features function, named as load_features_test

# Step 1: Benign Node Selection
### Identify candidate nodes with the same label as potentially malicious nodes.

In [ ]:
def split_malicious_nodes_by_label(malicious_indices, labels):

    r"""Groups malicious nodes based on their labels.

    :param malicious_indices: List of indices for malicious nodes.
    :param labels: List of labels for all nodes.
    :return: Dictionary with label as key and list of malicious node indices as values.
    """

    malicious_groups = defaultdict(list)

    for node_idx in malicious_indices:
        node_label = labels[node_idx]
        malicious_groups[node_label].append(node_idx)

    return dict(malicious_groups)


In [ ]:
def split_nodes_by_label(phrases, labels):

    r"""Groups all nodes based on their labels.

    :param phrases: List of phrases (interactions) for all nodes.
    :param labels: List of labels for all nodes.
    :return: Dictionary with label as key and list of node indices as values.
    """

    node_groups = defaultdict(list)

    for node_idx, node_features in enumerate(phrases):
        node_label = labels[node_idx]
        node_groups[node_label].append(node_idx)

    return dict(node_groups)


In [ ]:
def split_nodes_by_label_exclude_malicious(phrases, labels, indices_of_malicious_nodes):

    r"""Groups benign (non-malicious) nodes based on their labels.

    :param phrases: List of phrases (interactions) for all nodes.
    :param labels: List of labels for all nodes.
    :param indices_of_malicious_nodes: List of indices for malicious nodes.
    :return: Dictionary with label as key and list of benign node indices as values.
    """

    node_groups = defaultdict(list)
    malicious_set = set(indices_of_malicious_nodes)  

    for node_idx, node_features in enumerate(phrases):
        if node_idx not in malicious_set:
            node_label = labels[node_idx]
            node_groups[node_label].append(node_idx)

    return dict(node_groups)


# Step 2: Minimal Interaction Selection
### Prioritize nodes with the fewest interactions to reduce detectability.

In [ ]:
def filter_nodes_by_num_interactions(node_groups, phrases, min_len=15, max_len=100):

    r"""Filters the benign nodes in each label based on the number of interactions (phrase length).

    :param node_groups: Dictionary with label as key and list of benign node indices as values.
    :param phrases: List of phrases (interactions) for all nodes.
    :param min_len: Minimum number of interactions (hyperparameter).
    :param max_len: Maximum number of interactions (hyperparameter).
    :return: Dictionary with label as key and list of filtered benign node indices as values.
    """

    filtered_groups = {}

    for label, node_indices in node_groups.items():
        filtered_groups[label] = [
            idx for idx in node_indices if min_len <= len(phrases[idx]) <= max_len
        ]

    return filtered_groups


# Step 3: Contextual Consistency Filtering
### Choose nodes with higher cosine similarity to malicious nodes to preserve the contextual  integrity and stealth.

In [ ]:
def compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, nodes):

    r"""Orders the benign nodes within each label based on their contextual similarity to the malicious nodes in the same label in descending order.

    :param malicious_groups: Dictionary with label as key and list of malicious node indices as values.
    :param filtered_nodes_grouped: Dictionary with label as key and list of filtered benign node indices as values.
    :param nodes: Feature vectors (embeddings) for all nodes.
    :return: Dictionary with label as key and list of sorted benign node indices as values.
    """

    top_nodes_by_similarity = {}

    for label in malicious_groups.keys():
        if label in filtered_nodes_grouped:
            malicious_indices = malicious_groups[label]
            filtered_indices = filtered_nodes_grouped[label]

            # Get feature vectors for malicious and filtered nodes
            malicious_vectors = np.array([nodes[idx] for idx in malicious_indices])
            filtered_vectors = np.array([nodes[idx] for idx in filtered_indices])

            # Compute cosine similarity (malicious nodes vs filtered nodes)
            similarities = cosine_similarity(filtered_vectors, malicious_vectors)

            # Take the max similarity score for each filtered node
            max_similarities = similarities.max(axis=1)

            # Use np.argsort to get the indices that would sort the max_similarities array
            sorted_indices = np.array(filtered_indices)[np.argsort(-max_similarities)]  # Negative for descending order

            # Select top most similar nodes for the current label
            top_nodes_by_similarity[label] = sorted_indices

    return top_nodes_by_similarity


In [ ]:
def duplicate_and_replace(df, target_id, new_id):

    r"""Replaces given target id with new id

    :param target_id: The target id to be replaced
    :param new_id: The new id to replace the target id with
    :return: We return the new rows with new id's
    """

    # Find rows where target_id appears in actorID or objectID
    mask = df['actorID'].isin([target_id]) | df['objectID'].isin([target_id])
    selected_rows = df[mask]

    # Replace target_id with new_id in actorID and objectID using replace() method
    selected_rows[['actorID', 'objectID']] = selected_rows[['actorID', 'objectID']].replace(target_id, new_id)

    # Return the modified rows, no need to append to df within the function
    return selected_rows

In [ ]:
def load_features_test(dataframe, indices_of_flagged_nodes, xgb_cl): 

    r"""Function to apply evasion methodology if we supply with valid indices_of_flagged_nodes, else simply evalutes without evasion

    :param dataframe: The dataframe we evaluate on
    :param indices_of_flagged_nodes: List of indices for flagged (malicious) nodes.
    :param xgb_cl: Trained XGBoost classifier.
    :return: np.array(X_train), y_train, edges, mapping, malicious_and_benign
    """

    similarity_threshold=1

    # nodes is a list of nodes embeddings not phrases
    nodes, y_train, edges, mapping, label_map, node_entity_map, phrases_list = Featurize(dataframe)

    import numpy as np

    # Extract malicious phrases
    malicious_phrases = [phrases_list[i] for i in indices_of_flagged_nodes]

    # Compute phrase lengths directly (since they're lists already)
    lengths = np.array([len(p) for p in malicious_phrases])

    
    # Step 1: Benign Node Selection, Split all nodes by their labels excluding the malicious nodes
    all_nodes_grouped_excluding_malicious = split_nodes_by_label_exclude_malicious(phrases_list, y_train, indices_of_flagged_nodes)

    malicious_groups = split_malicious_nodes_by_label(indices_of_flagged_nodes, y_train) 

    # Step 2: Minimal Interaction Selection
    filtered_nodes_grouped = filter_nodes_by_num_interactions(all_nodes_grouped_excluding_malicious, phrases_list)

    # Step 3: Precompute cosine similarity between malicious nodes and benign candidates
    nodes = np.array(nodes)
    malicious_node_embeddings = nodes[indices_of_flagged_nodes]

    malicious_and_benign = {}
    iter = 1
    # Loop on each malicious node to find most similar benign node within same label group
    for idx, malicious_embedding in zip(indices_of_flagged_nodes, malicious_node_embeddings):
        iter += 1
        
        malicious_label = y_train[idx]

        benign_candidate_indices = filtered_nodes_grouped.get(malicious_label, [])

        if not benign_candidate_indices:
            continue  # Skip if no benign candidates available

        # Embeddings of benign candidates
        benign_embeddings = nodes[benign_candidate_indices]

        # Compute cosine similarity
        similarities = cosine_similarity(
            malicious_embedding.reshape(1, -1), benign_embeddings
        ).flatten()

        # Find the benign node with highest similarity
        max_sim_idx = np.argmax(similarities)
        most_similar_benign_node_idx = benign_candidate_indices[max_sim_idx]

        # Map malicious node ID to benign node ID
        malicious_node_ID = mapping[idx]
        benign_node_ID = mapping[most_similar_benign_node_idx]

        malicious_and_benign[malicious_node_ID] = benign_node_ID

        # Calculate top 10% count (at least 1)
        top_count = max(1, int(np.ceil(len(similarities) * 0.1)))

        # Store top 10% similar benign nodes for inspection
        top_n_indices = np.argsort(similarities)[::-1][:top_count]

        # top_benign_nodes_with_similarity = [
        #     (mapping[benign_candidate_indices[i]], similarities[i]) for i in top_n_indices
        # ]

        best_confidence = -1
        best_candidate_idx = None

        for i in top_n_indices:
            node_idx = benign_candidate_indices[i]
            node_features = nodes[node_idx].reshape(1, -1)

            # Pad with 20 zeros to match model's expected 40 features
            padding = np.zeros((1, 20))
            node_features_padded = np.hstack([node_features, padding])

            # Model prediction probabilities
            proba = xgb_cl.predict_proba(node_features_padded)[0]
            predicted_label = np.argmax(proba)
            true_label = y_train[node_idx]

            # Keep only if predicted correctly
            if predicted_label == true_label:
                confidence = proba[predicted_label]
                if confidence > best_confidence:
                    best_confidence = confidence
                    best_candidate_idx = node_idx

        # Fallback: if no correct predictions in top 10%, use most similar one
        if best_candidate_idx is None:
            best_candidate_idx = benign_candidate_indices[top_n_indices[0]]

        # Map malicious node ID to chosen benign node ID
        malicious_node_ID = mapping[idx]
        benign_node_ID = mapping[best_candidate_idx]
        malicious_and_benign[malicious_node_ID] = benign_node_ID

    # Prepare a list to collect the modified rows
    modified_rows = []

    triggered_actions = []
    
    # Now loop over the malicious node IDs and perform batch processing
    for malicious_node_ID, benign_node_ID in tqdm(malicious_and_benign.items(), desc="Processing Flagged Nodes"):
        selected_rows = duplicate_and_replace(dataframe, benign_node_ID, malicious_node_ID)
        triggered_actions.append(selected_rows)
        modified_rows.append(selected_rows)


    # After all iterations, concatenate all the new rows once
    df_curated = pd.concat([dataframe] + modified_rows, ignore_index=True)

    # Test Plausability
    unique_exec_action_path_df1 = dataframe[["objectname", "actorname", "action"]].drop_duplicates()
    unique_exec_action_path_df2 = df_curated[["objectname", "actorname", "action"]].drop_duplicates()
    if len(unique_exec_action_path_df1) == len(unique_exec_action_path_df2):
        print("Added Actions are plausible")
    else:
        print("Added Actions are not plausible")
        print("The number of unique actions are:", len(unique_exec_action_path_df2) - len(unique_exec_action_path_df1))

    print("The number of triggered events are:", len(df_curated) - len(dataframe))
    
    # nodes is node embedding not phrase
    nodes, y_train, edges, mapping, label_map, node_entity_map, phrases_list = Featurize(df_curated)
    
    X_train = []
    for i, map_id in enumerate(mapping):
        label = label_map[map_id]
        node_embedding = np.zeros(20)  
    
        if label in gnn_map:
            embedding, stored_set = gnn_map[label]
            current_set = node_entity_map[map_id]
            similarity_metric = len(current_set.intersection(stored_set)) / len(current_set.union(stored_set))

            if similarity_metric >= similarity_threshold:
                node_embedding = np.array(embedding)

        X_train.append(np.hstack((nodes[i], node_embedding)))

    return np.array(X_train), y_train, edges, mapping, malicious_and_benign

def evaluate_model(df, xgb_cl, confidence_threshold, indices_of_flagged_nodes):

    x, y, edges, mapp, malicious_and_benign  = load_features_test(df, indices_of_flagged_nodes, xgb_cl)

    pred = xgb_cl.predict(x)
    proba = xgb_cl.predict_proba(x) 

    sorted_proba = np.sort(proba, axis=1)
    conf = (sorted_proba[:, -1] - sorted_proba[:, -2]) / sorted_proba[:, -1]
    normalized_conf = (conf - conf.min()) / conf.max()

    check = (pred == y) & (normalized_conf > confidence_threshold)
    flag = ~torch.tensor(check)

    # index = utils.mask_to_index(flag).tolist()
    index = torch.nonzero(flag, as_tuple=False).view(-1).tolist()
    
    ids = {mapp[idx] for idx in index}

    return ids, edges, mapp, malicious_and_benign

### Evaluation Helper functions

In [ ]:
def Get_Adjacent(ids, mapp, edges, hops):
    if hops == 0:
        return set()
    neighbors = set()
    for edge in zip(edges[0], edges[1]):
        if any(mapp[node] in ids for node in edge): 
            neighbors.update(mapp[node] for node in edge)
    if hops > 1:
        neighbors = neighbors.union(Get_Adjacent(neighbors, mapp, edges, hops - 1))
    return neighbors

def calculate_metrics(TP, FP, FN, TN):
    FPR = FP / (FP + TN) if FP + TN > 0 else 0
    TPR = TP / (TP + FN) if TP + FN > 0 else 0
    TNR = TN / (TN + FP) if TN + FP > 0 else 0
    prec = TP / (TP + FP) if TP + FP > 0 else 0
    rec = TP / (TP + FN) if TP + FN > 0 else 0
    fscore = (2 * prec * rec) / (prec + rec) if prec + rec > 0 else 0
    return prec, rec, fscore, FPR, TPR, TNR

def helper(MP, all_pids, GP, edges, mapp):
    TP = MP.intersection(GP)  
    FP = MP - GP              
    FN = GP - MP              
    TN = all_pids - (GP | MP) 
    two_hop_gp = Get_Adjacent(GP, mapp, edges, 1)   
    two_hop_tp = Get_Adjacent(TP, mapp, edges, 1)     
    FPL = FP - two_hop_gp                            
    TPL = TP.union(FN.intersection(two_hop_tp))        
    FN = FN - two_hop_tp                      
    TP, FP, FN, TN = len(TPL), len(FPL), len(FN), len(TN)
    prec, rec, fscore, FPR, TPR, TNR = calculate_metrics(TP, FP, FN, TN)
    
    print(f"True Positives: {TP}, True Negatives: {TN}, False Positives: {FP}, False Negatives: {FN}")
    print(f"Precision: {round(prec, 2)}, Recall: {round(rec, 2)}, Fscore: {round(fscore, 2)}")
    print(f"False Positive Rate: {round(FPR, 2)}, True Positive Rate: {round(TPR, 2)}, True Negative Rate: {round(TNR, 2)}")

    return TP, TN, FP, FN, prec, rec, fscore


### Functions to load the datasets, ground truth from the .txt files and to load .pkl files

In [ ]:
def load_events_from_hosts(hosts):
    all_events = []
    for host in hosts:
        path = f'SysClient0{host}.systemia.com.txt'
        with open(path, 'r') as file:
            raw_events = [json.loads(line) for line in file]
        all_events.extend(raw_events)
    return all_events

def load_pkl(fname):
    with open(fname, 'rb') as f:
        obj = pickle.load(f)
    return obj

def load_ground_truth(gt_file):
    with open(gt_file, 'r') as file:
        ground_truth_nodes = set(file.read().split())
    return ground_truth_nodes

### Evaluating the 3 datasets before and after applying evasion steps to compare performance and evasion efficiency

### Testing Flash on OpTC Malicious Upgrade Attack OpTC 1

In [ ]:
all_events = load_events_from_hosts(['051'])
df_upgrade_attack = transform(all_events)

EnActIds = [x['actorID'] for x in all_events]
EnObjIds = [x['objectID'] for x in all_events]
EntitySet = set(EnActIds).union(set(EnObjIds))

ground_truth_nodes = load_ground_truth('optc.txt')
ground_truth_nodes = [x for x in ground_truth_nodes if x in EntitySet]
ground_truth_nodes = set(ground_truth_nodes)

print("Number of malicious nodes:", len(ground_truth_nodes))

### Before evasion steps

In [ ]:
indices_of_flagged_nodes = []

xgboost_model = load_pkl(xgboost_weights)
identified_ids, edges, mapp, malicious_and_benign  = evaluate_model(df_upgrade_attack, xgboost_model, 0.6, indices_of_flagged_nodes)
alerts = helper(identified_ids, EntitySet, ground_truth_nodes, edges, mapp)
print("Number of flagged nodes:", len(identified_ids))

s=0
for flagged_id in identified_ids:
    if flagged_id in ground_truth_nodes:
        s+=1
print("Number of nodes that are malicious and flagged", s, "\n")

indices_of_malicious_nodes = []
for id in ground_truth_nodes:
    if id in mapp:
        indices_of_malicious_nodes.append(mapp.index(id))

indices_of_flagged_nodes = []
for flagged_id in identified_ids:
    indices_of_flagged_nodes.append(mapp.index(flagged_id))

### Applying evasion steps and evaluating

In [ ]:
xgboost_model = load_pkl(xgboost_weights)
identified_ids, edges, mapp, malicious_and_benign = evaluate_model(df_upgrade_attack, xgboost_model, 0.6, indices_of_flagged_nodes)
alerts = helper(identified_ids, EntitySet, ground_truth_nodes, edges, mapp)
print("Number of flagged nodes:", len(identified_ids))
s=0
for flagged_id in identified_ids:
    if flagged_id in ground_truth_nodes:
        s+=1    
print("Number of nodes that are malicious and flagged", s)

### Testing Flash on OpTC Plain PowerShell Empire Attack OpTC 3

In [ ]:
all_events = load_events_from_hosts(['201'])

df_plain_powerShell_Empire_attack = transform(all_events)

EnActIds = [x['actorID'] for x in all_events]
EnObjIds = [x['objectID'] for x in all_events]
EntitySet = set(EnActIds).union(set(EnObjIds))

ground_truth_nodes = load_ground_truth('optc.txt')
ground_truth_nodes = [x for x in ground_truth_nodes if x in EntitySet]
ground_truth_nodes = set(ground_truth_nodes)

print("Number of malicious nodes:", len(ground_truth_nodes))

### Before evasion steps

In [ ]:
indices_of_flagged_nodes = []

xgboost_model = load_pkl(xgboost_weights)
identified_ids, edges, mapp, malicious_and_benign = evaluate_model(df_plain_powerShell_Empire_attack, xgboost_model, 0.6, indices_of_flagged_nodes)
alerts = helper(identified_ids, EntitySet, ground_truth_nodes, edges, mapp)
print("Number of flagged nodes:", len(identified_ids))
for flagged_id in identified_ids:
    if flagged_id in ground_truth_nodes:
        s+=1
print("Number of nodes that are malicious and flagged", s)

indices_of_malicious_nodes = []
for id in ground_truth_nodes:
    if id in mapp:
        indices_of_malicious_nodes.append(mapp.index(id))

indices_of_flagged_nodes = []
for flagged_id in identified_ids:
    indices_of_flagged_nodes.append(mapp.index(flagged_id))

### Applying evasion steps and evaluating

In [ ]:
xgboost_model = load_pkl(xgboost_weights)
identified_ids, edges, mapp, malicious_and_benign = evaluate_model(df_plain_powerShell_Empire_attack, xgboost_model, 0.6, indices_of_flagged_nodes)
alerts = helper(identified_ids, EntitySet, ground_truth_nodes, edges, mapp)
print("Number of flagged nodes:", len(identified_ids))
s=0
for flagged_id in identified_ids:
    if flagged_id in ground_truth_nodes:
        s+=1
print("Number of nodes that are malicious and flagged", s)

### Testing Flash on OpTC Custom PowerShell Empire Attack OpTC 2

In [ ]:
all_events = load_events_from_hosts(['501'])
df_custom_powerShell_Empire_attack = transform(all_events)

EnActIds = [x['actorID'] for x in all_events]
EnObjIds = [x['objectID'] for x in all_events]
EntitySet = set(EnActIds).union(set(EnObjIds))

ground_truth_nodes = load_ground_truth('optc.txt')
ground_truth_nodes = [x for x in ground_truth_nodes if x in EntitySet]
ground_truth_nodes = set(ground_truth_nodes)
print("Number of malicious nodes:", len(ground_truth_nodes))

### Before evasion steps

In [ ]:
indices_of_flagged_nodes = []

xgboost_model = load_pkl(xgboost_weights)
identified_ids, edges, mapp, malicious_and_benign = evaluate_model(df_custom_powerShell_Empire_attack, xgboost_model, 0.6, indices_of_flagged_nodes)
alerts = helper(identified_ids, EntitySet, ground_truth_nodes, edges, mapp)
print("Number of flagged nodes:", len(identified_ids))
for flagged_id in identified_ids:
    if flagged_id in ground_truth_nodes:
        s+=1
print("Number of nodes that are malicious and flagged", s)

indices_of_malicious_nodes = []
for id in ground_truth_nodes:
    if id in mapp:
        indices_of_malicious_nodes.append(mapp.index(id))

indices_of_flagged_nodes = []
for flagged_id in identified_ids:
    indices_of_flagged_nodes.append(mapp.index(flagged_id))

### Applying evasion steps and evaluating

In [ ]:
xgboost_model = load_pkl(xgboost_weights)
identified_ids, edges, mapp, malicious_and_benign = evaluate_model(df_custom_powerShell_Empire_attack, xgboost_model, 0.6, indices_of_flagged_nodes)
alerts = helper(identified_ids, EntitySet, ground_truth_nodes, edges, mapp)
print("Number of flagged nodes:", len(identified_ids))
s=0
for flagged_id in identified_ids:
    if flagged_id in ground_truth_nodes:
        s+=1
print("Number of nodes that are malicious and flagged", s)